# Sujet 4 : SVM

Le sujet s'inspire de la partie pratique du chapitre Méthode à noyau. Lancez la commande "jupyter-notebook" via un terminal et créez un nouveau notebook.

Pour le tp, vous aurez besoin de plusieurs bibliothèques de bases. Importez-les dès le début :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.datasets as skdata
import sklearn.preprocessing as skprep
import sklearn.svm as sksvm
import sklearn.model_selection as sksel

Nous utiliserons également une fonction permettant de tracer la frontière entre deux classes pour SVM...

In [ ]:
# clf : instance d'un classifieur
# window : coins de la fenêtre [minX, maxX, minY, maxY]
def plot_frontiere(clf, window):
    x0s = np.linspace(window[0], window[1], 100)
    x1s = np.linspace(window[2], window[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X).reshape(x0.shape)
    y_decision = clf.decision_function(X).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred,  alpha=0.2)
    plt.contourf(x0, x1, y_decision,  alpha=0.1)

Et une autre fonction pour SVR, la variante de SVM pour trouver une régression 

In [ ]:
# svm_reg : instance de SVR
# X : données
# y : vrais classes
# axes : coins de la fenêtre [minX, maxX, minY, maxY]
def plot_svm_regression(svm_reg, X, y, axes):
    x1s = np.linspace(axes[0], axes[1], 100).reshape(100, 1)
    y_pred = svm_reg.predict(x1s)
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$")
    plt.plot(x1s, y_pred + svm_reg.epsilon, "k--")
    plt.plot(x1s, y_pred - svm_reg.epsilon, "k--")
    plt.scatter(X[svm_reg.support_], y[svm_reg.support_], s=180, facecolors='#FFAAAA')
    plt.plot(X, y, "bo")
    plt.xlabel(r"$x_1$", fontsize=14)
    plt.ylabel(r"$y$", fontsize=14)
    plt.legend(loc="upper left", fontsize=14)
    plt.axis(axes)

## Pratique sur des données synthétiques

### SVM

On s'intéresse en premier à SVM. Pour étudier l'algorithme, on créé un jeu de données de données synthétique pour lequel on applique ensuite une standardisation. La standardisation permet d'obtenir pour chaque attribut la moyenne à 0 et l'écart type à 1.

In [ ]:
X1,y1=skdata.make_moons(n_samples=400,noise=0.15,random_state=42) # génération des données

X1norm=...      # standardisation

minX=np.floor(X1norm.min(axis=0)) # calcul du min (pour la suite)
maxX=...  # calcul du max (pour la suite)

Puis un SVM avec un noyau polynomial et un coefficient $\theta=1$ est appliqué sur l'ensemble du jeu de données.

In [ ]:
svm_model=sksvm.SVC(...) # creation du model SVM avec les paramètres voulus
... # utilisation des données sur le modèle
plot_frontiere(svm_model,[minX[0],maxX[0],minX[1],maxX[1]]) # affichage des frontières
... # affichage des points

1) Par défaut, quel est le degré appliqué au noyau polynomial ? Quel est la valeur du terme de régularisation ?

Vous allez maintenant jouer avoir les paramètres pour pouvoir étudier leurs comportements.

2) Que se passe-t'il si on augmente le terme de régularisation ?

3) Que donne un degré égal à 1 ? Que se passe-t'il si on augmente le degré du polynôme ?

4) Testez maintenant un noyau gaussien. Quel est le comportement de son paramètre $\gamma$ ?


In [ ]:
...

### SVR

L'algorithme SVM a également été adapté à la régression. Tester SVR (Support Vector Regression) sur les données synthétiques suivantes : 

In [ ]:
np.random.seed(42)
n = 50
X2 = 6 * np.random.rand(n, 1)
y2 = (1.5* np.sin(X2) + X2 + np.random.randn(n,1)*0.7).ravel()

plt.plot(X2, y2,"bo");

Trouvez les paramètres les plus adaptés pour un noyau polynomial et un noyau gaussien.

In [ ]:
svr_model_linear=sksvm.SVR(kernel='linear').fit(X2,y2)
svr_model_rbf=sksvm.SVR(...).fit(X2,y2)
svr_model_poly=sksvm.SVR(...).fit(X2,y2)

minX=np.floor(np.min(X2))
maxX=np.ceil(np.max(X2))
xtest=np.linspace(minX,maxX,100).reshape(100, 1)
plt.plot(xtest,svr_model_linear.predict(xtest),'-r')
plt.plot(xtest,svr_model_rbf.predict(xtest),'-m')
plt.plot(xtest,svr_model_poly.predict(xtest),'-g')
plt.legend(['linear','rbf','poly'])
plt.plot(X2, y2,"bo");


Un SVR linéaire est principalement défini par le paramètre $\varepsilon$, qui spécifie la largeur du tube dans lequel aucune pénalité n'est associée.

5) Quel est cette largeur par défaut ?

6) Tester d'autres valeur de largeur du tube. Cela change-t'il les résultats ? (aidez vous de la fonction plot_svm_regression pour visualiser les résultats)


In [ ]:
...

In [ ]:
...

## Pratique sur des données réelles

Vous pouvez maintenant tester SVR sur un jeu de données réel. Nous utiliserons pour cette partie la fonction suivante :

In [ ]:
def plotImages(xtrain,xtest,ytrain,yfound,im):
    
    plt.subplot(131)
    plt.imshow(np.reshape(im,(16,16)),cmap='gray')
    
    imTrain=np.ones((16,16))
    for i in range(0,xtrain.shape[0]):
        imTrain[xtrain[i][1],xtrain[i][0]]=ytrain[i]
    
    plt.subplot(132)
    plt.imshow(imTrain,cmap='gray')
    
    imTest=imTrain
    for i in range(0,xtest.shape[0]):
        imTest[xtest[i][1],xtest[i][0]]=yfound[i]
    
    plt.subplot(133)
    plt.imshow(imTest,cmap='gray')
    plt.show(block=False)

Récupérez maintenant la base USPS sur l’ENT et chargez la :

In [ ]:
uspsXapp = np.load('USPS/uspsXapp.npy')
uspsYapp = np.load('USPS/uspsYapp.npy')

Cette base contient plus de 7000 images de chiffres manuscrits. Pour visualiser une image, utilisez le code suivant

In [ ]:
nimg = 456
im = uspsXapp[:,nimg]
lab = uspsYapp[nimg]-1

plt.imshow(np.reshape(im,(16,16)));
plt.title('Training: %i' %lab);

Notez que le choix d’utiliser la première image est complètement arbitraire est vous pouvez donc utiliser n'importe laquelle pour la suite du TP. L'objectif est de reconstruire une image donc des pixels sont perdus.

Pour cela, on va tout d'abord construire la base de donnéees. On considère que les donnéees sont les coordonnées des pixels et les étiquettes les valeurs des pixels. Il faut donc construire une matrice X de dimension 2*nbPixels contenant tous les couples de coordonnéees. Y sera un vecteur contenant toutes les valeurs des pixels correspondant aux coordonnéees.

In [ ]:
[x1,x2] = np.meshgrid(range(16),range(16))
trainvecT = np.concatenate((np.reshape(x1,(16*16,1)),np.reshape(x2,(16*16,1))),axis=1)
trainlabT = im

Découper la base créée en une base d'apprentissage et une base de tests. Ici il n’y a pas de répartition particulière à respecter comme dans la classification. Les points placés dans la base de test correspondent aux pixels manquants.

In [ ]:
tx=0.1 # pourcentage d'individus dans le test
xtrain,xtest,ytrain,ytest=sksel.train_test_split(...) # mettre une graine pour l'aléatoire

Faire un premier essai de regression, puis faire varier le taux de pixels manquants (donc augmenter ou diminuer la taille de l'ensemble d'apprentissage) afin d’observer la robustesse de la méthode.

In [ ]:
# SVR avec noyau gaussien
svr_rbf = sksvm.SVR(...)
yfound = ...

# plot figures
plotImages(xtrain,xtest,ytrain,yfound,im)



Mettre en place une validation croisée afin de définir les meilleurs paramètres pour un SVR avec un noyau gaussien.


On utilise maintenant le même jeu de données mais pour effectuer de la classification. La base USPS contient 10 classes : les chiffres de 0 à 9. Pour cette partie, le SVM va être utilisé en mode classique, à savoir la classification binaire.
1. Constituer une base comprenant 300 exemples de 2 classes (au choix, mais évitez le 4 contre 9 qui est particulièrement difficile!), soit 600 exemples. Définissez également le vecteur y. Les exemples restants des deux classes formeront la base de test.

2. Mettre en place une validation croisée afin de de déterminer le meilleur couple de paramètres (σ,C) pour le noyau Gaussien. Donner la performance en test.

3. Affichez les exemples mal classées.

4. Observation de l’influence des paramètres (sans validation croisée) :
    • Fixez C selon la valeur optimale de la question précédente. Faites varier σ et affichez l'évolution de la performance en apprentissage et en test ainsi que le temps d'apprentissage. Commentez.
    • Fixez σ selon la valeur optimale de la question précédente. Faites varier C et afficher l'évolution de la performance en apprentissage et en test ainsi que le temps d'apprentissage. Commentez.